# panggil LLM

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def generate_text_manual(
    model_name: str = "Qwen/Qwen3-4B-Instruct-2507",
    prompt: str = "What is the capital of France?",
    max_new_tokens: int = 50,
    device: str = "cpu",
):
    """
    Generate text dengan manual loop (token-by-token).
    Berguna untuk observasi per-step atau custom logic.
    
    Parameters:
    -----------
    model_name : str
        HuggingFace model identifier
    prompt : str
        Input text prompt
    max_new_tokens : int
        Maximum tokens to generate
    device : str
        Device to run on ("cuda" or "cpu")
    
    Returns:
    --------
    str : Generated text
    """
    # Load model dan tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
        device_map=device,
    )
    model.eval()
    
    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]  # shape: [1, seq_len]
    print("Input IDs:", input_ids)
    
    generated_ids = input_ids.clone()
    
    # Manual generation loop
    with torch.no_grad():
        for step in range(max_new_tokens):
            # Forward pass
            outputs = model(
                input_ids=generated_ids,
                use_cache=False,  # Tidak pakai KV-cache (full recomputation setiap step)
            )
            print("outputs awal: ", outputs)
            
            # Get logits untuk token berikutnya
            next_token_logits = outputs.logits[:, -1, :]  # [1, vocab_size]
            print(f"{step}next logit: {next_token_logits}")
            
            # Sample next token (greedy decoding)
            next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)  # [1, 1]
            print(f"{step}next token id: {next_token_id}")

            # Append to sequence
            generated_ids = torch.cat([generated_ids, next_token_id], dim=1)
            print(f"{step}generated ids: {generated_ids}")
            
            # Check for EOS token
            if next_token_id.item() == tokenizer.eos_token_id:
                break
    
    # Decode
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return generated_text


# Contoh penggunaan
text = generate_text_manual(
    prompt="sebutkan apa saja faktor penggerak harga crypto?",
    max_new_tokens=10,
)
print(text)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Input IDs: tensor([[  325,  8088,  8656, 60593, 78786, 66792,   269, 36686,  1389,   585,
         71348, 19028,    30]])
outputs awal:  CausalLMOutputWithPast(loss=None, logits=tensor([[[ 1.1172,  3.3125,  2.7656,  ..., -3.5625, -3.5625, -3.5625],
         [ 4.4375,  4.7188, -0.2656,  ..., -6.9375, -6.9375, -6.9375],
         [ 3.7344,  1.6406, -1.4922,  ..., -7.6875, -7.6875, -7.6875],
         ...,
         [ 2.6719,  0.7734, -0.0742,  ..., -9.2500, -9.2500, -9.2500],
         [ 5.8438,  3.9531,  3.5469,  ..., -6.6250, -6.6250, -6.6250],
         [ 2.1094, -1.3125,  1.4141,  ..., -8.0000, -8.0000, -8.0000]]],
       dtype=torch.bfloat16), past_key_values=None, hidden_states=None, attentions=None)
0next logit: tensor([[ 2.1094, -1.3125,  1.4141,  ..., -8.0000, -8.0000, -8.0000]],
       dtype=torch.bfloat16)
0next token id: tensor([[9101]])
0generated ids: tensor([[  325,  8088,  8656, 60593, 78786, 66792,   269, 36686,  1389,   585,
         71348, 19028,    30,  9101]])
outputs awa

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def generate_text_optimized(
    model_name: str = "Qwen/Qwen2.5-7B-Instruct",
    messages: list = None,
    max_new_tokens: int = 256,
    temperature: float = 1,
    device: str = "cpu",
):
    """
    Generate text menggunakan model.generate() (built-in optimizations).
    Sudah include KV-cache, sampling strategies, dll.
    
    Parameters:
    -----------
    model_name : str
        HuggingFace model identifier
    messages : list
        Chat messages format: [{"role": "user", "content": "..."}]
    max_new_tokens : int
        Maximum new tokens to generate
    temperature : float
        Sampling temperature (0.0 = greedy, higher = more random)
        - 0.0-0.3: Deterministic, focused
        - 0.7-1.0: Balanced creativity
        - 1.5+: Very creative, less coherent
    top_p : float (0.0-1.0)
        Nucleus sampling threshold
        - 0.95: Keep 95% probability mass (recommended)
        - 1.0: Consider all tokens
    device : str
        "cuda" or "cpu"
    
    Returns:
    --------
    str : Generated text
    """
    # Load model dan tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map=device,
    )
    model.eval()
    
    # Default messages jika tidak diberikan
    if messages is None:
        messages = [
            {"role": "user", "content": "What is the capital of France?"}
        ]
    
    # Apply chat template (untuk instruction-tuned models)
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    print("input awal: ", inputs)
    print("input awal id: ", inputs.input_ids)
    
    # Generate dengan optimizations
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,  # Enable sampling jika temperature > 0
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,  # ← PENTING: Enable KV-cache untuk efisiensi
        )
        print("outputs generate: ", outputs)
    
    # Decode hanya token baru (skip prompt)
    prompt_length = inputs.input_ids.shape[1]
    print("prompt length: ", prompt_length)
    
    generated_ids = outputs[0][prompt_length:]
    print("generated ids: ", generated_ids)
    
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    
    return generated_text


# Contoh penggunaan
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "jelaskan secara singkat faktor penggerak harga crypto."},
]

text = generate_text_optimized(
    messages=messages,
)
print(text)

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 42.84it/s]


input awal:  {'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,     73,    301,  78351,  72327,
           7780,  33755,  66792,    269,  36686,   1389,    585,  71348,  19028,
             13, 151645,    198, 151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1]])}
input awal id:  tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,     73,    301,  78351,  72327,
           7780,  33755,  66792,    269,  36686,   1389,    585,  71348,  19028,
             13, 151645,    198, 151644,  77091,    198]])
outputs generate:  tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,     73,    301,  78351,  72327,
           7780,  33755,  66792, 

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Tuple, List

def extract_hidden_states(
    model_name: str = "Qwen/Qwen2.5-7B-Instruct",
    prompt: str = "What is machine learning?",
    device: str = "cpu",
) -> Tuple[str, torch.Tensor, List[torch.Tensor]]:
    """
    Extract hidden states dari semua layer untuk setiap token.
    
    Parameters:
    -----------
    model_name : str
        HuggingFace model identifier
    prompt : str
        Input text
    device : str
        Device to run on
    
    Returns:
    --------
    generated_text : str
        Generated output text
    last_hidden_state : torch.Tensor
        Hidden state dari layer terakhir, token terakhir
        Shape: [hidden_dim] (e.g., [4096] untuk Qwen2.5-7B)
    all_hidden_states : List[torch.Tensor]
        List of hidden states dari semua layer
        Length: num_layers + 1 (termasuk embedding layer)
        Each shape: [batch=1, seq_len, hidden_dim]
    """
    # Load model dan tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print("tokenizer: ", tokenizer)
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map=device,
    )
    print("model: ", model)
    model.eval()
    print("model eval: ", model.eval())
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Generate dengan hidden states output
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
            output_hidden_states=True,  # ← KUNCI: Enable hidden states output
            return_dict_in_generate=True,  # ← Return dictionary dengan metadata
        )
    
    # Decode text
    prompt_length = inputs.input_ids.shape[1]
    generated_ids = outputs.sequences[0][prompt_length:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # Extract hidden states
    # outputs.hidden_states = tuple of tuples
    # Structure: (num_generated_tokens, num_layers, batch, seq_len, hidden_dim)
    
    # Hidden state dari token terakhir yang di-generate
    # -1 = token terakhir, -1 = layer terakhir, 0 = batch pertama, -1 = posisi terakhir
    last_token_hidden = outputs.hidden_states[-1][-1][0, -1, :]  # [hidden_dim]
    
    # Semua hidden states dari semua layer untuk token terakhir
    all_layer_hidden = [
        layer_output[0, -1, :]  # [hidden_dim] untuk setiap layer
        for layer_output in outputs.hidden_states[-1]  # Ambil dari token terakhir
    ]
    
    print(f"Generated text: {generated_text}")
    print(f"Last hidden state shape: {last_token_hidden.shape}")
    print(f"Number of layers: {len(all_layer_hidden)}")
    print(f"Each layer shape: {all_layer_hidden[0].shape}")
    
    return generated_text, last_token_hidden, all_layer_hidden


# Contoh penggunaan
text, last_hidden, all_hidden = extract_hidden_states(
    prompt="apa itu bitcoin?",
    device="cpu",
)

# Serialize hidden state untuk penyimpanan (seperti di Fase 1B)
import io
buffer = io.BytesIO()
torch.save(last_hidden.cpu(), buffer)
embedding_bytes = buffer.getvalue()
print(f"Serialized size: {len(embedding_bytes)} bytes")

tokenizer:  Qwen2TokenizerFast(name_or_path='Qwen/Qwen2.5-7B-Instruct', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_w

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 14.86it/s]


model:  Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)

# panggil LLM + monitor

In [2]:
import sys
import os

# Mendapatkan path folder 'ai-agent' (parent dari folder 'llm')
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [3]:
print(parent_dir)

/root/projects/first-experiment/ai-agent


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from debug.monitor import Monitor, MonitorConfig

In [8]:
import os
print(os.getcwd())

/root/projects/first-experiment/ai-agent/llm


### Generate text dengan full monitoring ke file log.

In [ ]:
def generate_text_monitored(
    model_name: str = "Qwen/Qwen3-4B-Instruct-2507",
    prompt: str = "What is AI?",
    max_new_tokens: int = 10,
    device: str = "cpu",
    session_name: str = "llm_generation",
):
    """
    Generate text dengan full monitoring ke file log.
    Console hanya menampilkan progress, detail di log file.
    """
    
    # Setup monitor: console untuk progress, file untuk detail
    config = MonitorConfig(
        log_dir="./logs",
        console_output=True,  # Progress di console
        file_output=True,     # Detail di file
        max_tensor_elements=50,
        tensor_precision=4,
    )
    
    with Monitor(config, session_name=session_name) as monitor:
        
        # ══════════════════════════════════════════════════════════════
        # INITIALIZATION
        # ══════════════════════════════════════════════════════════════
        
        with monitor.step("Model Loading", to_console=True):
            monitor.log("Loading tokenizer...", to_console=True, to_file=True)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            
            monitor.log("Loading model...", to_console=True, to_file=True)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                dtype=torch.bfloat16,
                device_map=device,
            )
            model.eval()
            monitor.log("✓ Model ready", to_console=True)
        
        # ══════════════════════════════════════════════════════════════
        # TOKENIZATION
        # ══════════════════════════════════════════════════════════════
        
        with monitor.step("Tokenization"):
            monitor.log_variable("prompt", prompt, to_console=False)
            
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            input_ids = inputs["input_ids"]
            
            # Log detail ke file saja
            monitor.log_tensor("input_ids", input_ids, to_console=False)
            monitor.log_variable("input_length", input_ids.shape[1], to_console=True)
            
            # Decode tokens untuk visualisasi
            tokens_text = [tokenizer.decode([tid]) for tid in input_ids[0]]
            monitor.log_variable("input_tokens", tokens_text, to_console=False)
        
        # ══════════════════════════════════════════════════════════════
        # GENERATION LOOP
        # ══════════════════════════════════════════════════════════════
        
        generated_ids = input_ids.clone()
        generated_tokens_list = []
        
        with monitor.step("Generation Loop", to_console=True):
            
            for step in range(max_new_tokens):
                
                # Progress ke console, detail ke file
                monitor.log(f"Step {step+1}/{max_new_tokens}", to_console=True)
                
                with monitor.step(f"Step_{step}", to_console=False):
                    
                    # ──────────────────────────────────────────────────
                    # Forward Pass
                    # ──────────────────────────────────────────────────
                    
                    with monitor.step("Forward_Pass", to_console=False):
                        with torch.no_grad():
                            outputs = model(
                                input_ids=generated_ids,
                                use_cache=False,
                            )
                        
                        # Log outputs detail ke file
                        monitor.log_variable("outputs_type", type(outputs).__name__, to_console=False)
                        monitor.log_tensor("logits", outputs.logits, max_elements=30, to_console=False)
                        monitor.log_variable("logits_shape", list(outputs.logits.shape), to_console=False)
                    
                    # ──────────────────────────────────────────────────
                    # Extract Next Token Logits
                    # ──────────────────────────────────────────────────
                    
                    next_token_logits = outputs.logits[:, -1, :]
                    
                    # Log logits stats ke file
                    monitor.log_variable("next_logits_shape", list(next_token_logits.shape), to_console=False)
                    monitor.log_variable("next_logits_min", next_token_logits.min().item(), to_console=False)
                    monitor.log_variable("next_logits_max", next_token_logits.max().item(), to_console=False)
                    monitor.log_variable("next_logits_mean", next_token_logits.mean().item(), to_console=False)
                    
                    # Top-5 candidates (untuk analisis)
                    top5_logits, top5_indices = torch.topk(next_token_logits[0], k=5)
                    top5_tokens = [tokenizer.decode([idx]) for idx in top5_indices]
                    
                    top5_info = {
                        "indices": top5_indices.tolist(),
                        "logits": [f"{l:.4f}" for l in top5_logits.tolist()],
                        "tokens": top5_tokens,
                    }
                    monitor.log_dict("top5_candidates", top5_info, to_console=False)
                    
                    # ──────────────────────────────────────────────────
                    # Greedy Decoding
                    # ──────────────────────────────────────────────────
                    
                    next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
                    next_token_text = tokenizer.decode(next_token_id[0])
                    
                    monitor.log_variable("selected_token_id", next_token_id.item(), to_console=False)
                    monitor.log_variable("selected_token_text", next_token_text, to_console=True)
                    
                    generated_tokens_list.append(next_token_text)
                    
                    # ──────────────────────────────────────────────────
                    # Append Token
                    # ──────────────────────────────────────────────────
                    
                    generated_ids = torch.cat([generated_ids, next_token_id], dim=1)
                    monitor.log_variable("generated_ids_shape", list(generated_ids.shape), to_console=False)
                    
                    # ──────────────────────────────────────────────────
                    # Check EOS
                    # ──────────────────────────────────────────────────
                    
                    if next_token_id.item() == tokenizer.eos_token_id:
                        monitor.log("EOS token reached, stopping generation", to_console=True)
                        break
        
        # ══════════════════════════════════════════════════════════════
        # FINAL OUTPUT
        # ══════════════════════════════════════════════════════════════
        
        with monitor.step("Decoding Final Output"):
            generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
            
            monitor.log_separator()
            monitor.log_variable("generated_text", generated_text, to_console=True)
            monitor.log_separator()
            
            # Summary ke file
            summary = {
                "prompt": prompt,
                "generated_tokens": generated_tokens_list,
                "total_tokens": len(generated_tokens_list),
                "final_text": generated_text,
            }
            monitor.log_dict("generation_summary", summary, to_console=False)
    
    print(f"\n✓ Full log saved to: {monitor.log_file_path}")
    print(f"✓ Check the log file for detailed tensor values and statistics")
    
    return generated_text

In [9]:
print("\n[1] Monitored Generation (detail in logs/)")
print("-" * 60)
text = generate_text_monitored(
    prompt="Apa itu machine learning?",
    max_new_tokens=256,
    device="cpu",
)
print(f"Generated: {text}")


[1] Monitored Generation (detail in logs/)
------------------------------------------------------------
[   0.000s] ────────────────────────────────────────────────────────────
[   0.001s] ▶ START: Model Loading
[   0.001s]   Loading tokenizer...
[   2.598s]   Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[   4.076s]   ✓ Model ready
[   4.077s] ◀ END: Model Loading (took 4.075s)
[   4.077s] ────────────────────────────────────────────────────────────
[   4.077s] ▶ START: Tokenization
[   4.082s]   input_length = 6
[   4.082s] ◀ END: Tokenization (took 0.005s)
[   4.082s] ────────────────────────────────────────────────────────────
[   4.082s] ▶ START: Generation Loop
[   4.082s]   Step 1/256
[  15.807s]     selected_token_text =  Ap
[  15.819s]   Step 2/256
[  23.428s]     selected_token_text = a
[  23.433s]   Step 3/256
[  34.777s]     selected_token_text =  per
[  34.798s]   Step 4/256
[  45.944s]     selected_token_text = bed
[  45.947s]   Step 5/256
[  56.982s]     selected_token_text = aan
[  56.985s]   Step 6/256
[  62.545s]     selected_token_text =  ant
[  62.551s]   Step 7/256
[  68.329s]     selected_token_text = ara
[  68.349s]   Step 8/256
[  73.582s]     selected_token_text =  machine
[  73.585s]   Step 9/256
[  77.784s]     selected_token_text =  learning
[  77.788s]   Ste

### Extract hidden states dengan monitoring detail ke file.

In [7]:
def extract_hidden_states_monitored(
    model_name: str = "Qwen/Qwen2.5-7B-Instruct",
    prompt: str = "Explain AI briefly:",
    device: str = "cpu",
):
    """
    Extract hidden states dengan monitoring detail ke file.
    """
    
    config = MonitorConfig(
        log_dir="./logs",
        console_output=True,
        file_output=True,
        max_tensor_elements=20,  # Limit karena hidden states besar
    )
    
    with Monitor(config, session_name="hidden_states_extraction") as monitor:
        
        with monitor.step("Setup", to_console=True):
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.bfloat16,
                device_map=device,
            )
            model.eval()
        
        with monitor.step("Generation with Hidden States"):
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=10,
                    output_hidden_states=True,
                    return_dict_in_generate=True,
                )
            
            # Log hidden states structure
            monitor.log_variable("num_generated_tokens", len(outputs.hidden_states), to_console=True)
            
            # Analyze last token's hidden states
            last_token_hidden = outputs.hidden_states[-1]
            monitor.log_variable("num_layers", len(last_token_hidden), to_console=True)
            
            # Log detail setiap layer ke file
            with monitor.step("Layer Analysis", to_console=False):
                for layer_idx, layer_output in enumerate(last_token_hidden):
                    monitor.log_tensor(
                        f"layer_{layer_idx}_output",
                        layer_output,
                        max_elements=10,
                        to_console=False,
                    )
            
            # Extract final hidden state
            final_hidden = last_token_hidden[-1][0, -1, :]  # [hidden_dim]
            monitor.log_tensor("final_hidden_state", final_hidden, to_console=False)
            
            # Summary
            monitor.log_separator()
            monitor.log(f"✓ Extracted hidden states from {len(last_token_hidden)} layers", to_console=True)
            monitor.log(f"✓ Final hidden state shape: {final_hidden.shape}", to_console=True)
    
    print(f"\n✓ Detailed analysis saved to: {monitor.log_file_path}")
    return final_hidden

### Compare manual loop vs model.generate() dengan monitoring.

In [8]:
def compare_generation_methods(
    model_name: str = "Qwen/Qwen2.5-7B-Instruct",
    prompt: str = "List three colors:",
    device: str = "cpu",
):
    """
    Compare manual loop vs model.generate() dengan monitoring.
    """
    
    config = MonitorConfig(
        log_dir="./logs",
        console_output=True,
        file_output=True,
    )
    
    with Monitor(config, session_name="method_comparison") as monitor:
        
        # Load model
        with monitor.step("Model Loading"):
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.bfloat16,
                device_map=device,
            )
            model.eval()
        
        # Method 1: Manual loop
        with monitor.step("Method 1: Manual Loop", to_console=True):
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            
            import time
            start = time.time()
            
            generated_ids = inputs["input_ids"].clone()
            for step in range(5):
                with torch.no_grad():
                    outputs = model(input_ids=generated_ids, use_cache=False)
                next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
                generated_ids = torch.cat([generated_ids, next_token], dim=1)
            
            text1 = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
            time1 = time.time() - start
            
            monitor.log_variable("output_text", text1, to_console=False)
            monitor.log_variable("time_taken", f"{time1:.4f}s", to_console=True)
        
        # Method 2: model.generate()
        with monitor.step("Method 2: model.generate()", to_console=True):
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            
            start = time.time()
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=5,
                    use_cache=True,
                )
            
            text2 = tokenizer.decode(outputs[0], skip_special_tokens=True)
            time2 = time.time() - start
            
            monitor.log_variable("output_text", text2, to_console=False)
            monitor.log_variable("time_taken", f"{time2:.4f}s", to_console=True)
        
        # Comparison
        monitor.log_separator()
        monitor.log(f"Manual Loop: {time1:.4f}s", to_console=True)
        monitor.log(f"model.generate(): {time2:.4f}s", to_console=True)
        monitor.log(f"Speedup: {time1/time2:.2f}x", to_console=True)
        
        comparison = {
            "manual_time": time1,
            "generate_time": time2,
            "speedup": time1 / time2,
            "manual_output": text1,
            "generate_output": text2,
        }
        monitor.log_dict("comparison_results", comparison, to_console=False)
    
    print(f"\n✓ Comparison saved to: {monitor.log_file_path}")